# **(Data Collection)**

## Objectives

* Fetch Dataset for Fetus Framework from Kaggle and save as raw data, prepare it for further process

## Inputs

* Kaggle Dataset contains 1528 2D sagital-view ultrasound images of 1519 females collected from Shenzhen People’s Hospital. Data was used for standard/non-standard (S-NS) plane classification, a key step for NT measurement and Down Syndrome assessment.

## Outputs

*  Generate Dataset:inputs/Dataset/Standard and Non-standard_dataset

 



In [1]:
import numpy

import os

---

# Change working directory

* Assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

You need to change the working directory from its current folder to its parent folder
* Access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'c:\\python_projects\\ML_2\\jupyter_notebooks'

Make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\python_projects\\ML_2'

In [5]:
#install Kaggle 
!pip install kaggle

In [6]:
os.environ['KAGGLE_CONFIG_DIR']=os.getcwd()


In [7]:
import os
from pathlib import Path

# Kaggle dataset identifier
KaggleDatasetPath = "https://www.kaggle.com/datasets/orvile/dataset-for-fetus-framework"

# Destination folder
DestinationFolder = Path(r"C:\python_projects\ML_2\inputs\Dataset")

# Create directory if it does not exist
DestinationFolder.mkdir(parents=True, exist_ok=True)

# Download dataset
!kaggle datasets download -d {KaggleDatasetPath} -p "{DestinationFolder}"

Invalid dataset specification https://www.kaggle.com/datasets/orvile/dataset-for-fetus-framework


In [8]:
KaggleDatasetPath = "https://www.kaggle.com/datasets/orvile/dataset-for-fetus-framework"
DestinationFolder = "inputs/Dataset_Clean"

os.makedirs(DestinationFolder, exist_ok=True)
! kaggle datasets download -d {KaggleDatasetPath} -p {DestinationFolder}

Invalid dataset specification https://www.kaggle.com/datasets/orvile/dataset-for-fetus-framework


In [9]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: c:\python_projects\ML_2


In [10]:
DestinationFolder = os.path.abspath("inputs/Dataset_Clean")
zip_file_path = os.path.join(DestinationFolder, "Dataset.zip")

In [11]:
import zipfile
DestinationFolder = os.path.abspath("inputs/Dataset_Clean")
zip_file_path = os.path.join(DestinationFolder, "Dataset_Clean.zip")

if not os.path.exists(zip_file_path):
    print(f"Error: File not found at {zip_file_path}")
else:
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(DestinationFolder)
        print(f"Extracted: {zip_file_path}")
        
        os.remove(zip_file_path)  # Remove the ZIP file after extraction
        print(f"Deleted: {zip_file_path}")
    except Exception as e:
        print(f"Error extracting ZIP file: {e}")

Error: File not found at c:\python_projects\ML_2\inputs\Dataset_Clean\Dataset_Clean.zip


Data Cleaning
 Check and remove non-image files

In [12]:
def remove_non_image_file(my_data_dir):
    image_extenstion=('.png','.jpg','.jpeg', '.bmp')
    folders=os.listdir(my_data_dir)
    for folder in folders:
        files=os.listdir(my_data_dir + '/' + folder)
        # print (files)
        i=[]
        j=[]
        for given_file in files:
            if not given_file.lower().endswith(image_extenstion):
                file_location=my_data_dir + '/' + folder + '/' + given_file
                os.remove(file_location) #remove non image file
                i.append(1)
            else:
                j.append(1)
        print(f"Folder:{folder} - has image file", len(j))
        print(f"Folder:{folder} - has image file", len(i))
        


In [13]:
from pathlib import Path

dataset_dir = Path(r"C:\python_projects\ML_2\inputs\Dataset_Clean\validation")

extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

total = 0

for class_name in ["Standard", "Non-standard"]:
    folder = dataset_dir / class_name

    images = [
        f for f in folder.rglob("*")
        if f.is_file() and f.suffix.lower() in extensions
    ]

    print(f"{class_name}: {len(images)} images")
    total += len(images)

print("-" * 35)
print(f"TOTAL BEFORE SPLITTING: {total} images")

Standard: 229 images
Non-standard: 93 images
-----------------------------------
TOTAL BEFORE SPLITTING: 322 images


In [ ]:
import hashlib, pathlib
import numpy as np
from PIL import Image

root = pathlib.Path (r"C:\python_projects\ML_2\inputs\Dataset\validation")
exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
files = [p for p in root.rglob("*") if p.suffix.lower() in exts]

def top_folder(p):
    return p.relative_to(root).parts[0]

def dhash(p, size=16):
    im = Image.open(p).convert("L").resize((size + 1, size))
    a = np.asarray(im, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

md5s, hashes = [], []
for p in files:
    md5s.append(hashlib.md5(p.read_bytes()).hexdigest())
    hashes.append(dhash(p))
H = np.stack(hashes)

# 1) counts per top-level folder
from collections import Counter
print(Counter(top_folder(p) for p in files))

# 2) exact duplicates
by_md5 = {}
for p, m in zip(files, md5s):
    by_md5.setdefault(m, []).append(p)
exact = [v for v in by_md5.values() if len(v) > 1]
print("exact duplicate groups:", len(exact))

# 3) near duplicates (Hamming distance <= 6 of 256 bits)
D = (H[:, None, :] != H[None, :, :]).sum(-1)
i, j = np.where(np.triu(D <= 6, k=1))
cross = [(files[a], files[b]) for a, b in zip(i, j) if top_folder(files[a]) != top_folder(files[b])]
print("near-duplicate pairs across folders:", len(cross))
print(Counter((top_folder(a), top_folder(b)) for a, b in cross))

ValueError: need at least one array to stack

In [ ]:
remove_non_image_file(
    my_data_dir=r"C:\python_projects\ML_2\inputs\Dataset_Clean"
)

Folder:Non-standard - has image file 0
Folder:Non-standard - has image file 0
Folder:Standard - has image file 0
Folder:Standard - has image file 0


PermissionError: [WinError 5] Åtkomst nekad: 'C:\\python_projects\\ML_2\\inputs\\Dataset_Clean/test/Non-Standard'

Split train validation test set 

In [16]:
!pip install joblib


In [17]:
import shutil
import random

In [18]:
import os
import shutil
import random
import joblib

def split_train_validation_test_images(my_data_dir, train_set_ratio, validation_set_ratio, test_set_ratio):

    if train_set_ratio + validation_set_ratio + test_set_ratio != 1.0:
        print("train_set_ratio + validation_set_ratio + test_set_ratio should sum to 1.0")
        return

    # gets classes labels
    labels = os.listdir(my_data_dir)  # it should get only the folder name
    if 'test' in labels:
        pass
    else:
        # create train, test folders with classes labels sub-folder
        for folder in ['train', 'validation', 'test']:
            for label in labels:
                os.makedirs(name=my_data_dir + '/' + folder + '/' + label)

        for label in labels:

            files = os.listdir(my_data_dir + '/' + label)
            random.shuffle(files)

            train_set_files_qty = int(len(files) * train_set_ratio)
            validation_set_files_qty = int(len(files) * validation_set_ratio)
            count = 1
            for file_name in files:
                if count <= train_set_files_qty:
                    # move a given file to the train set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/train/' + label + '/' + file_name)

                elif count <= (train_set_files_qty + validation_set_files_qty):
                    # move a given file to the validation set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/validation/' + label + '/' + file_name)

                else:
                    # move given file to test set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/test/' + label + '/' + file_name)

                count += 1

            os.rmdir(my_data_dir + '/' + label)

            

In [19]:
import os
import random
import shutil

def split_dataset(
    my_data_dir,
    train_set_ratio=0.7,
    validation_set_ratio=0.1
):
    images = [
        f for f in os.listdir(my_data_dir)
        if os.path.isfile(os.path.join(my_data_dir, f))
        and f.lower().endswith(('.png', '.jpg', '.jpeg', '.dcm'))
    ]

    random.seed(42)
    random.shuffle(images)

    n_train = int(len(images) * train_set_ratio)
    n_val = int(len(images) * validation_set_ratio)

    splits = {
        "train": images[:n_train],
        "validation": images[n_train:n_train+n_val],
        "test": images[n_train+n_val:]
    }

    for folder, filenames in splits.items():
        output_dir = os.path.join(my_data_dir, folder)
        os.makedirs(output_dir, exist_ok=True)

        for filename in filenames:
            shutil.copy2(
                os.path.join(my_data_dir, filename),
                os.path.join(output_dir, filename)
            )

    for folder, filenames in splits.items():
        print(f"{folder}: {len(filenames)} images")

In [20]:
split_dataset(
    my_data_dir='inputs/Dataset_Clean',
    train_set_ratio=0.7,
    validation_set_ratio=0.1
)

train: 0 images
validation: 0 images
test: 0 images


In [21]:
import os
import shutil
import random

my_data_dir = 'inputs/Dataset_Clean'

classes = ['Standard', 'Non-Standard']
splits = ['train', 'validation', 'test']
split_ratios = [0.7, 0.1, 0.2]

random.seed(42)

def split_and_copy_images():

    # Check that class folders exist
    for cls in classes:
        src_folder = os.path.join(my_data_dir, cls)

        if not os.path.isdir(src_folder):
            raise FileNotFoundError(
                f"Class folder not found: {src_folder}"
            )

    # Create destination folders
    for split in splits:
        for cls in classes:
            split_path = os.path.join(
                my_data_dir, split, cls
            )
            os.makedirs(split_path, exist_ok=True)

    # Split each class
    for cls in classes:

        src_folder = os.path.join(my_data_dir, cls)

        all_images = [
            f for f in os.listdir(src_folder)
            if os.path.isfile(os.path.join(src_folder, f))
            and f.lower().endswith(
                ('.png', '.jpg', '.jpeg')
            )
        ]

        random.shuffle(all_images)

        total = len(all_images)

        train_end = int(total * split_ratios[0])
        val_end = train_end + int(total * split_ratios[1])

        train_files = all_images[:train_end]
        val_files = all_images[train_end:val_end]
        test_files = all_images[val_end:]

        for split, file_list in zip(
            splits,
            [train_files, val_files, test_files]
        ):
            for filename in file_list:

                src_path = os.path.join(
                    src_folder, filename
                )

                dst_path = os.path.join(
                    my_data_dir, split, cls, filename
                )

                shutil.copy2(src_path, dst_path)

        print(
            f"{cls}: "
            f"{len(train_files)} train, "
            f"{len(val_files)} validation, "
            f"{len(test_files)} test"
        )

split_and_copy_images()

Standard: 0 train, 0 validation, 0 test
Non-Standard: 0 train, 0 validation, 0 test


In [22]:
import os
import shutil
import random

my_data_dir='inputs/Dataset_Clean'
classes=['Standard', 'Non-Standard']
splits=['train', 'validation', 'test']
split_ratios=[0.7,0.1,0.2]

# Make sure destination folders exist
for split in splits:
    for cls in classes:
        split_path = os.path.join(my_data_dir, split, cls)
        os.makedirs(split_path, exist_ok=True)

    # Function to split and move images
def split_and_move_images():
    for cls in classes:
        src_folder = os.path.join(my_data_dir, cls)
        all_images = [f for f in os.listdir(src_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        random.shuffle(all_images)

        total = len(all_images)
        train_end = int(total * split_ratios[0])
        val_end = train_end + int(total * split_ratios[1])

        train_files = all_images[:train_end]
        val_files = all_images[train_end:val_end]
        test_files = all_images[val_end:]

        for split, file_list in zip(splits, [train_files, val_files, test_files]):
            for filename in file_list:
                src_path = os.path.join(src_folder, filename)
                dst_path = os.path.join(my_data_dir, split, cls, filename)
                shutil.move(src_path, dst_path)

        print(f"{cls}: {len(train_files)} train, {len(val_files)} validation, {len(test_files)} test")

split_and_move_images()

Standard: 0 train, 0 validation, 0 test
Non-Standard: 0 train, 0 validation, 0 test


---

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* If you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.